### Language Identification -- Building Statistical Classifiers

Zoey Liu

In this notebook, we will play around with building statistical classifiers to do language identification using part-of-speech (POS) tags.

In [ ]:
# Loading some packages
# Make sure to install the packages if you run into errors
# !pip install PACKAGE_NAME
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

I picked (simplified) Mandarin Chinese, because! And also Basque --- the endlessly interesting "alien language". We would be only using the output files for both here (each line of the output file contains the POS tag sequence for the corresponding line of the input file).

Let's start with looking at our data.

In [ ]:
!head POS_data/UD_Basque/train.output

In [ ]:
!head POS_data/UD_Chinese/train.output

In [ ]:
## Loading the data
def load_pos_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            tag_list = line.strip().split()
            data.append(tag_list)
    return data

In [ ]:
# Basque data
basque_pos_data = load_pos_data('POS_data/UD_Basque/train.output')

# Chinese data
chinese_pos_data = load_pos_data('POS_data/UD_Chinese/train.output')

Now we have our **text data**, we need to extract some features to build the classifiers. Recall that these features are subjective --- you can decide the feature set and extract whatever you'd like.

We will start with some frequency/probability distribution.

In [ ]:
# We first need a function to identify all the unique POS tags in a given dataset
def get_unique_pos_tags(data):
    unique_tags = set()
    for sentence in data:
        unique_tags.update(sentence)
    return sorted(list(unique_tags))

In [ ]:
# We then convert the POS tag sequences to feature vectors
def pos_tags_to_prob(pos_sequence, unique_tags):
    """Convert POS tag sequence to feature vector (normalized distribution)"""
    total = len(pos_sequence)
    counts = Counter(pos_sequence)
    features = [counts.get(tag, 0) / total for tag in unique_tags]
    return features

In [ ]:
# Extracting probability distribution features
basque_unique_tags = get_unique_pos_tags(basque_pos_data)
chinese_unique_tags = get_unique_pos_tags(chinese_pos_data)

# Combining unique tags of the two languages
# QUESTION: WHY?
combined_unique_tags = sorted(list(set(basque_unique_tags).union(set(chinese_unique_tags))))

basque_features = [pos_tags_to_prob(sent, combined_unique_tags) for sent in basque_pos_data]
chinese_features = [pos_tags_to_prob(sent, combined_unique_tags) for sent in chinese_pos_data]

# Let's take a look at the features
for i in range(3):
    print(f"Basque sentence {i+1} features: {basque_features[i]}")

print('\n')
print('\n')
for i in range(3):
    print(f"Chinese sentence {i+1} features: {chinese_features[i]}")

# For classifications, recall that we often work with arrays/matrices
# This will be the input, X, to the classifiers
basque_X_features = np.array(basque_features)
chinese_X_features = np.array(chinese_features)

In [ ]:
# Creating arrays for the labels, y
# Let's set Basque as 0 and Chinese as 1
# But this is arbitrary, you can choose whatever you'd like
basque_y = np.zeros(len(basque_X_features))
chinese_y = np.ones(len(chinese_X_features))

In [ ]:
# Combine the data
X = np.vstack((basque_X_features, chinese_X_features))
y = np.concatenate((basque_y, chinese_y))

In [ ]:
# Create DataFrame for visualization to see if we have everything right
df = pd.DataFrame(X, columns=combined_unique_tags)
df['language'] = ['Basque'] * len(basque_features) + ['Chinese'] * len(chinese_features)

print("="*60)
print("BINARY CLASSIFICATION: ENGLISH vs JAPANESE")
print("="*60)
print(f"\nDataset shape: {X.shape}")
print(f"Number of features (POS tags): {X.shape[1]}")
print(f"Basque samples: {len(basque_features)}")
print(f"Chinese samples: {len(chinese_features)}")


In [ ]:
# Visualize POS tag distributions
fig, axes = plt.subplots(1, 1, figsize=(16, 5))

# Average distribution for each language
avg_basque = basque_X_features.mean(axis=0)
avg_chinese = chinese_X_features.mean(axis=0)

x_pos = np.arange(len(combined_unique_tags))
width = 0.35

axes.bar(x_pos - width/2, avg_basque, width, label='Basque', alpha=0.8, color='blue')
axes.bar(x_pos + width/2, avg_chinese, width, label='Chinese', alpha=0.8, color='green')
axes.set_xlabel('POS Tags')
axes.set_ylabel('Average Proportion')
axes.set_title('Average POS Tag Distribution by Language')
axes.set_xticks(x_pos)
axes.set_xticklabels(combined_unique_tags, rotation=45, ha='right')
axes.legend()
axes.grid(axis='y', alpha=0.3)

For any model training and evaluation, we need a training set and a test set. In this case we will randomly split each POS feature set into training and test setes.

In [ ]:
# Randomly split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10, stratify=y
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
# Train logistic regression
lr_binary = LogisticRegression(random_state=1, max_iter=1000)
lr_binary.fit(X_train, y_train)

# Predictions
y_pred = lr_binary.predict(X_test)
y_pred_proba = lr_binary.predict_proba(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Basque', 'Chinese']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Basque', 'Chinese'],
            yticklabels=['Basque', 'Chinese'])
plt.title('Confusion Matrix - Binary Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

QUESTIONS: 

(1) What do the results tell you?

(2) How would we know the models have *learned* something?

In [ ]:
# Majority Class Baseline
from sklearn.dummy import DummyClassifier

majority_clf = DummyClassifier(strategy='most_frequent', random_state=6)
majority_clf.fit(X_train, y_train)
y_pred_majority = majority_clf.predict(X_test)
acc_majority = accuracy_score(y_test, y_pred_majority)
print(f"\nMajority Class Baseline: {acc_majority:.3f}")

QUESTION: What is 0.652?

In [ ]:
round(88 / (88+47), 3)

Explainability is important. Period. What features help the most with distinguishing between Basque vs. Chinese?

In [ ]:
# Feature importance
coefficients = lr_binary.coef_[0]
feature_importance = pd.DataFrame({
    'POS_tag': combined_unique_tags,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

print("\nFeature Importance (Coefficients):")
print(feature_importance)

plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'purple' for c in feature_importance['coefficient']]
plt.barh(feature_importance['POS_tag'], feature_importance['coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient (Positive → Basque, Negative → Chinese)')
plt.title('POS Tag Coefficients for Language Classification')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

**Exercise:** Use the data from `UD_Arabic` and `UD_Turkish` to build a logistic regression classifier that does multi-class classification.